# Image selectivity analysis

Split out from `figure_3_supplemental.ipynb`. Lifetime sparseness, tuning-curve heatmaps, and population tuning curves. Contains shared setup plus the image-selectivity section.

### Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import seaborn as sns
sns.set_context('notebook', font_scale=1.5, rc={'lines.markeredgewidth': 2})

# Statistical analysis
import statsmodels.formula.api as smf
from statsmodels.formula.api import ols
from scipy.stats import chi2

# Data access
from visual_behavior.data_access import loading as loading
import visual_behavior.data_access.utilities as utilities
from allensdk.brain_observatory.behavior.behavior_project_cache import VisualBehaviorOphysProjectCache

# Utilities
import visual_behavior.visualization.utils as utils
import visual_behavior.visualization.ophys.platform_paper_figures as ppf

%load_ext autoreload
%autoreload 2
%matplotlib inline

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/opt/miniconda3/envs/visual_behavior_sdk/lib/python3.9/site-packages/traitlets/traitlets.py", line 632, in get
    value = obj._trait_values[self.name]
KeyError: '_control_lock'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/miniconda3/envs/visual_behavior_sdk/lib/python3.9/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/opt/miniconda3/envs/visual_behavior_sdk/lib/python3.9/site-packages/ipykernel/kernelbase.py", line 301, in dispatch_control
    async with self._control_lock:
  File "/opt/miniconda3/envs/visual_behavior_sdk/lib/python3.9/site-packages/traitlets/traitlets.py", line 687, in __get__
    return t.cast(G, self.get(obj, cls))  # the G should encode the Optional
  File "/opt/miniconda3/envs/visual_behavior_sdk/lib/python3.9/site-packages/traitlets/traitlets.p

### Configuration

In [2]:
# When True, skip heavy-duty loading + slow-MLM heatmaps, exposure,
# active/passive, epoch, and image-selectivity analyses (fast iteration).
skip_slow_plots = False

save_dir = os.path.join(loading.get_figures_save_dir(), 'platform_paper_figures', 'figure_3_supplemental')
folder = 'population_activity'

# Create save directories if they don't exist
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

if not os.path.exists(os.path.join(save_dir, folder)):
    os.makedirs(os.path.join(save_dir, folder))

palette = utils.get_experience_level_colors()
experience_levels = utils.get_new_experience_levels()
experience_level_colors = utils.get_experience_level_colors()
cell_types = utils.get_cell_types()

### Data Loading

#### Metadata tables

In [3]:
# AllenSDK cache
platform_cache_dir = loading.get_platform_analysis_cache_dir()
cache = loading._get_cache()

# Curated metadata tables
metadata_tables_dir = loading.get_metadata_tables_dir()
experiments_table = pd.read_csv(os.path.join(metadata_tables_dir, 'all_ophys_experiments_table.csv'), index_col=0)
platform_experiments = pd.read_csv(os.path.join(metadata_tables_dir, 'platform_paper_ophys_experiments_table.csv'), index_col=0)
platform_cells_table = pd.read_csv(os.path.join(metadata_tables_dir, 'platform_paper_ophys_cells_table.csv'), index_col=0)
behavior_sessions = pd.read_csv(os.path.join(metadata_tables_dir, 'platform_behavior_sessions_table.csv'), index_col=0)
matched_cells_table = pd.read_csv(os.path.join(metadata_tables_dir, 'platform_paper_matched_ophys_cells_table.csv'), index_col=0)

# Relevant IDs
matched_cells = matched_cells_table.cell_specimen_id.unique()
matched_experiments = matched_cells_table.ophys_experiment_id.unique()

# Cre lines, cell types, and palette for plotting
cre_lines = np.sort(platform_cells_table.cre_line.unique())
cell_types = utils.get_cell_types()
experience_levels = utils.get_experience_levels()
palette = utils.get_experience_level_colors()


/opt/miniconda3/envs/visual_behavior_sdk/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)
/opt/miniconda3/envs/visual_behavior_sdk/lib/python3.9/site-packages/allensdk/brain_observatory/behavior/behavior_project_cache/behavior_project_cache.py:135: UpdatedStimulusPresentationTableWarning: 
	As of AllenSDK version 2.16.0, the latest Visual Behavior Ophys data has been significantly updated from previous releases. Specifically the user will need to update all processing of the stimulus_presentations tables. These tables now include multiple stimulus types delineated by the columns `stimulus_block` and `stimulus_bl

#### Load data tables

In [4]:
data_type = 'filtered_events'
session_subset = 'full_session'
inclusion_criteria = 'platform_experiment_table'

metrics_dir = loading.get_cell_metrics_dir()
filepath = os.path.join(metrics_dir, f'merged_model_free_metrics_table_{data_type}_{inclusion_criteria}.csv')
filtered_events_metrics_table = pd.read_csv(filepath, index_col=0)

metrics_table = filtered_events_metrics_table.copy()

# Merge with platform experiments
metrics_table = metrics_table.merge(platform_experiments.reset_index(),
                                   on=['ophys_experiment_id', 'experience_level'])


# # Create event-type specific dataframes
# image_mdf = metrics_table[(metrics_table.event_type == 'images')]
# change_mdf = metrics_table[(metrics_table.event_type == 'changes')]
# omission_mdf = metrics_table[(metrics_table.event_type == 'omissions')]

# # Create each_image specific dataframe
# each_image_mdf = metrics_table[(metrics_table.event_type == 'images') &
#                                (metrics_table.image_id.notna())].copy()

In [5]:
data_type = 'events'
interpolate = True
output_sampling_rate = 30
inclusion_criteria = 'platform_experiment_table'


In [6]:

# Image changes
event_type = 'all'
conditions = ['cell_specimen_id', 'is_change']
is_change_mdf = loading.get_multi_session_df_for_conditions(
    data_type, event_type, conditions, inclusion_criteria,
    interpolate=interpolate, output_sampling_rate=output_sampling_rate,
    epoch_duration_mins=None)

change_mdf = is_change_mdf[is_change_mdf.is_change == True]
image_mdf = is_change_mdf[is_change_mdf.is_change == False]

# Omissions
conditions = ['cell_specimen_id', 'omitted']
omission_mdf = loading.get_multi_session_df_for_conditions(
    data_type, event_type, conditions, inclusion_criteria,
    interpolate=interpolate, output_sampling_rate=output_sampling_rate,
    epoch_duration_mins=None)

omission_mdf = omission_mdf[omission_mdf.omitted == True]

# Per image
conditions = ['cell_specimen_id', 'is_change', 'image_name']
each_image_mdf = loading.get_multi_session_df_for_conditions(
    data_type, event_type, conditions, inclusion_criteria,
    interpolate=interpolate, output_sampling_rate=output_sampling_rate,
    epoch_duration_mins=None)

/opt/miniconda3/envs/visual_behavior_sdk/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)
/opt/miniconda3/envs/visual_behavior_sdk/lib/python3.9/site-packages/allensdk/brain_observatory/behavior/behavior_project_cache/behavior_project_cache.py:135: UpdatedStimulusPresentationTableWarning: 
	As of AllenSDK version 2.16.0, the latest Visual Behavior Ophys data has been significantly updated from previous releases. Specifically the user will need to update all processing of the stimulus_presentations tables. These tables now include multiple stimulus types delineated by the columns `stimulus_block` and `stimulus_bl

1936
loading files from /Users/marinag/Library/CloudStorage/Dropbox/JupyterNotebooks/FinalDataAssetsforCodeOcean/visual_behavior_ophys_multi_session_dfs
mean_response_df_events_all_is_change_platform_experiment_table.pkl
loading multi_session_df from saved feather file at /Users/marinag/Library/CloudStorage/Dropbox/JupyterNotebooks/FinalDataAssetsforCodeOcean/visual_behavior_ophys_multi_session_dfs/mean_response_df_events_all_is_change_platform_experiment_table.feather
there are 1881 experiments in the full multi_session_df
there are 400 experiments in the multi_session_df after limiting to platform experiments
there are 400 experiments after filtering for inclusion criteria -  platform_experiment_table


/opt/miniconda3/envs/visual_behavior_sdk/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)
/opt/miniconda3/envs/visual_behavior_sdk/lib/python3.9/site-packages/allensdk/brain_observatory/behavior/behavior_project_cache/behavior_project_cache.py:135: UpdatedStimulusPresentationTableWarning: 
	As of AllenSDK version 2.16.0, the latest Visual Behavior Ophys data has been significantly updated from previous releases. Specifically the user will need to update all processing of the stimulus_presentations tables. These tables now include multiple stimulus types delineated by the columns `stimulus_block` and `stimulus_bl

1936
loading files from /Users/marinag/Library/CloudStorage/Dropbox/JupyterNotebooks/FinalDataAssetsforCodeOcean/visual_behavior_ophys_multi_session_dfs
mean_response_df_events_all_omitted_platform_experiment_table.pkl
loading multi_session_df from saved feather file at /Users/marinag/Library/CloudStorage/Dropbox/JupyterNotebooks/FinalDataAssetsforCodeOcean/visual_behavior_ophys_multi_session_dfs/mean_response_df_events_all_omitted_platform_experiment_table.feather
there are 1879 experiments in the full multi_session_df
there are 401 experiments in the multi_session_df after limiting to platform experiments
there are 401 experiments after filtering for inclusion criteria -  platform_experiment_table


/opt/miniconda3/envs/visual_behavior_sdk/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)
/opt/miniconda3/envs/visual_behavior_sdk/lib/python3.9/site-packages/allensdk/brain_observatory/behavior/behavior_project_cache/behavior_project_cache.py:135: UpdatedStimulusPresentationTableWarning: 
	As of AllenSDK version 2.16.0, the latest Visual Behavior Ophys data has been significantly updated from previous releases. Specifically the user will need to update all processing of the stimulus_presentations tables. These tables now include multiple stimulus types delineated by the columns `stimulus_block` and `stimulus_bl

1936
loading files from /Users/marinag/Library/CloudStorage/Dropbox/JupyterNotebooks/FinalDataAssetsforCodeOcean/visual_behavior_ophys_multi_session_dfs
mean_response_df_events_all_is_change_image_name_platform_experiment_table.pkl
loading multi_session_df from saved feather file at /Users/marinag/Library/CloudStorage/Dropbox/JupyterNotebooks/FinalDataAssetsforCodeOcean/visual_behavior_ophys_multi_session_dfs/mean_response_df_events_all_is_change_image_name_platform_experiment_table.feather
there are 1885 experiments in the full multi_session_df
there are 401 experiments in the multi_session_df after limiting to platform experiments
there are 401 experiments after filtering for inclusion criteria -  platform_experiment_table


## Image selectivity

Lifetime sparseness (a measure of image selectivity, computed as the kurtosis of
the distribution of responses across the 8 images, normalized by image count)
for non-change and change image presentations, across experience levels and cell
types. Shown for all cells and for the subset of cells with significant image
responses. Higher values indicate stronger tuning for specific images.
Corresponds to PPT Figure population_activity and supplemental image selectivity panels.

In [ ]:
if not skip_slow_plots:
    # all cells 

    metric = 'lifetime_sparseness_images'
    ylabel = 'Image selectivity - all cells'

    event_type = 'images'

    ylims = [-0.1, 1.01]

    ppf.plot_metric_distribution_by_experience(metrics_table, metric, plot_type='boxplot', horiz=False,
                                               add_zero_line=False, event_type=event_type, data_type=data_type,
                                               ylabel=ylabel, ylims=ylims, save_dir=save_dir, ax=None)


In [ ]:
if not skip_slow_plots:
    # responsive cells 

    metric = 'lifetime_sparseness_images'
    ylabel = 'Image selectivity of responsive cells (non-changes)'

    event_type = 'images'

    ylims = [-0.1, 1.01]

    tmp = metrics_table[metrics_table.fraction_significant_p_value_gray_screen_pref_image>0.1]

    ppf.plot_metric_distribution_by_experience(tmp, metric, plot_type='boxplot', horiz=False,
                                               add_zero_line=False, event_type=event_type, data_type=data_type,
                                               ylabel=ylabel, ylims=ylims, save_dir=save_dir, ax=None)

    # ppf.plot_metric_distribution_all_conditions(tmp, metric, event_type, data_type, ylabel=ylabel, ylims=ylims, 
    #                                             add_zero_line=False, save_dir=save_dir)

In [ ]:
if not skip_slow_plots:

    # responsive cells 

    metric = 'lifetime_sparseness_changes'
    ylabel = 'Image selectivity - changes, all cells'

    event_type = 'changes'

    ylims = [-0.1, 1.01]

    tmpt = metrics_table.copy()

    ppf.plot_metric_distribution_by_experience(tmp, metric, plot_type='boxplot', horiz=False,
                                               add_zero_line=False, event_type=event_type, data_type=data_type,
                                               ylabel=ylabel, ylims=ylims, save_dir=save_dir, ax=None)

In [ ]:
if not skip_slow_plots:

    # responsive cells 

    metric = 'lifetime_sparseness_changes'
    ylabel = 'Image selectivity of responsive cells (changes)'

    event_type = 'changes'

    ylims = [-0.1, 1.01]

    tmpt = metrics_table[metrics_table.fraction_significant_p_value_gray_screen_changes>0.1]

    ppf.plot_metric_distribution_by_experience(tmp, metric, plot_type='boxplot', horiz=False,
                                               add_zero_line=False, event_type=event_type, data_type=data_type,
                                               ylabel=ylabel, ylims=ylims, save_dir=save_dir, ax=None)

#### Tuning curve heatmaps

Heatmaps of mean response to each of the 8 images for all cells, with cells sorted
by their preferred image. Shown separately for non-change and change image
presentations across experience levels, illustrating how the population's image
preference structure changes (or is maintained) across Familiar, Novel, and Novel+
sessions.

##### non-changes

In [ ]:
if not skip_slow_plots:
    df = each_image_mdf[each_image_mdf.project_code=='VisualBehaviorMultiscope']
    df = df[(df.is_change==False)]
    row_condition = 'cell_type'
    col_condition = 'experience_level'


    ppf.plot_tuning_curve_heatmaps_for_conditioskip_slow_plotsns(df, data_type, 
                                              row_condition, col_condition, vmax=None, 
                                              cbar=False, cbar_label='Mean response',
                                              save_dir=save_dir, folder='image_tuning', suffix='_images', ax=None)

##### changes

In [ ]:
if not skip_slow_plots:
    df = each_image_mdf[each_image_mdf.project_code=='VisualBehaviorMultiscope']
    df = df[df.is_change==True]
    row_condition = 'cell_type'
    col_condition = 'experience_level'

    ppf.plot_tuning_curve_heatmaps_for_conditions(df, data_type, 
                                              row_condition, col_condition, vmax=None, 
                                              cbar=False, cbar_label='Mean response',
                                              save_dir=save_dir, folder='image_tuning', suffix='_changes', ax=None)

In [ ]:
# df = each_image_mdf[each_image_mdf.project_code=='VisualBehaviorMultiscope']
# df = df[df.is_change==True]
# row_condition = 'cell_type'
# col_condition = 'experience_level'

# plot_tuning_curve_heatmaps_for_conditions(df, data_type, 
#                                           row_condition, col_condition, vmax=None, 
#                                           cbar=True, cbar_label='Mean response',
#                                           save_dir=save_dir, folder='image_tuning', suffix='_changes_cbar', ax=None)

#### Population tuning curves

Population-averaged tuning curves showing the normalized mean response to each
image (sorted by response magnitude) across experience levels. Images are rank-
ordered by each cell's preferred-to-least-preferred for both non-change and change
presentations, then averaged across cells. The slope of these curves reflects the
degree of image selectivity at the population level.

In [ ]:
if not skip_slow_plots:
    df = each_image_mdf[each_image_mdf.project_code=='VisualBehaviorMultiscope']
    df = df[(df.is_change==False) & (df.image_name!='omitted')]
    responsive_cells = df[df.fraction_significant_p_value_gray_screen>0.1].cell_specimen_id.unique()
    df = df[df.cell_specimen_id.isin(responsive_cells)]
    print(len(df.cell_specimen_id.unique()))

    row_condition = 'cell_type'
    col_condition = 'experience_level'

    ppf.plot_population_tuning_curve_for_conditions(df, data_type, 
                                              row_condition, col_condition, ylabel='Normalized image response',
                                              save_dir=save_dir, folder='image_tuning', suffix='_images', ax=None)

In [ ]:
if not skip_slow_plots:
    df = each_image_mdf[each_image_mdf.project_code=='VisualBehaviorMultiscope']
    df = df[df.is_change==True]
    responsive_cells = df[df.fraction_significant_p_value_gray_screen>0.1].cell_specimen_id.unique()
    df = df[df.cell_specimen_id.isin(responsive_cells)]
    print(len(df.cell_specimen_id.unique()))

    row_condition = 'cell_type'
    col_condition = 'experience_level'

    ppf.plot_population_tuning_curve_for_conditions(df, data_type, row_condition, col_condition, ylabel='Normalized change response',
                                              save_dir=save_dir, folder='image_tuning', suffix='_changes', ax=None)